In [ ]:
import fitz  # PyMuPDF

def update_invoice(input_pdf, output_pdf, new_vendor_data, items):
    doc = fitz.open(input_pdf)
    page = doc[0]  # Focus on the first page

    # 1. Define areas to update (Coordinates need adjustment based on your specific PDF)
    # format: (x0, y0, x1, y1)
    vendor_name_area = (50, 50, 300, 100) 
    trn_area = (50, 120, 200, 140)

    # 2. Redact the old information
    page.add_redact_annot(vendor_name_area, fill=(1, 1, 1)) # White out
    page.add_redact_annot(trn_area, fill=(1, 1, 1))
    page.apply_redactions()

    # 3. Insert new text
    page.insert_text((50, 70), new_vendor_data['name'], fontsize=12, color=(0, 0, 0))
    page.insert_text((50, 130), f"TRN: {new_vendor_data['trn']}", fontsize=10)

    # 4. Update Line Items (Example for one row)
    # You would loop through your 'items' list and update corresponding coordinates
    page.insert_text((50, 400), items[0]['desc'], fontsize=9)

    doc.save(output_pdf)
    doc.close()

# Example Data for your 5 test cases
vendor_list = [
    {"name": "Gulf Hydraulic Solutions", "trn": "310998877600003"},
    {"name": "Desert Flow Systems", "trn": "310112233400003"},
    # ... add more
]

# Run the update
# update_invoice("Posted Sales Invoices (AEFF).pdf", "Test_Invoice_1.pdf", vendor_list[0], [])

In [1]:
! pip install pymupdf

   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.2 MB 2.0 MB/s eta 0:00:10
   -- ------------------------------------- 1.3/19.2 MB 1.8 MB/s eta 0:00:10
   --- ------------------------------------ 1.6/19.2 MB 1.6 MB/s eta 0:00:11
   --- ------------------------------------ 1.8/19.2 MB 1.7 MB/s eta 0:00:11
   ---- ----------------------------------- 2.1/19.2 MB 1.7 MB/s eta 0:00:10
   ---- ----------------------------------- 2.4/19.2 MB 1.6 MB/s eta 0:00:11
   ----- ---------------------------------- 2.6/19.2 MB 1.6 MB/s eta 0:00:11
   ----- ---------------------------------- 2.9/19.2 MB 1.5 MB/s eta 0:00:12
   ------- -------------------------------- 3.4/19.2 MB 1.5 MB/s eta 0:00:11
   ------- -------------------------------- 3.7/19.2 MB 1.6 MB/s eta 0:00:10
   -------- --------

In [23]:
import pdfplumber
import re
from chonkie import TableChunker

def log_info(msg):
    # Simple logger replacement
    print(msg)

def clean_cell(cell):
    """Recursively normalize a table cell to a string and handle percentages."""
    if cell is None:
        return ""
    if isinstance(cell, list):
        # Flatten any nested lists recursively
        return " ".join(clean_cell(c) for c in cell)
    # At this point, cell is a string/number
    cell_str = str(cell).strip()
    # Replace percentages
    cell_str = re.sub(r'(\d+(\.\d+)?)\s*%', r'\1-Percentage', cell_str)
    return cell_str

def Document_Extraction_from_Fitz_for_tp(pdf_path):
    html = """
    <html>
    <head>
    <meta charset="UTF-8">
    <style>
    body { font-family: Arial, sans-serif; white-space: pre-wrap; }
    table { border-collapse: collapse; width: 100%; margin: 8px 0; }
    td, th { border: 1px solid #ccc; padding: 4px; text-align: left; }
    </style>
    </head>
    <body>
    """
    pdf = None
    try:
        pdf = pdfplumber.open(pdf_path)
        for page in pdf.pages:
            # --- Extract tables ---
            for table in page.extract_tables() or []:
                html += "<table>\n"
                for row in table:
                    html += "<tr>" + "".join(f"<td>{(cell or '').strip()}</td>" for cell in row) + "</tr>\n"
                html += "</table>\n"

            # --- Extract text per line ---
            text = page.extract_text()  # cleaner than extract_words
            if text:
                clean_text = re.sub(r'\n+', '<br>', text)  # replace multiple newlines
                html += f"<p>{clean_text}</p>\n"

    except Exception as e:
        log_info(f"❌ Error processing PDF '{pdf_path}': {e}")
    finally:
        if pdf:
            pdf.close()
    html += "</body></html>"
    return html

In [25]:
Document_Extraction_from_Fitz_for_tp(r"C:\Users\BRADSOL\Downloads\ICCU250700002431-F6L6WW-001.pdf")

'\n    <html>\n    <head>\n    <meta charset="UTF-8">\n    <style>\n    body { font-family: Arial, sans-serif; white-space: pre-wrap; }\n    table { border-collapse: collapse; width: 100%; margin: 8px 0; }\n    td, th { border: 1px solid #ccc; padding: 4px; text-align: left; }\n    </style>\n    </head>\n    <body>\n    <table>\n<tr><td>Tax Invoice\nAIR INDIA EXPRESS LIMITED\n(Original For Recipient)\nAddress : 2nd Floor, Airlines House, NSCBI Airport, 39 C\nR Avenue Kolkata, Kolkata, West Bengal, 700052\nGSTN : 19AABCA0522B1ZH\nInvoice Number : ICCU250700002431\nInvoice Date : 02-07-2025\nPNR No : F6L6WW\nFlight No : 2826 Flight Date : 28-07-2025\nPassenger Name : Atanu Ghosh\nPassenger Address :\nFlight From : CCU Flight To : HYD\nGSTIN of Customer : 36AAACB7873P1Z0\nPlace of Supply : TELANGANA [36]\nGSTIN Customer Name : BIOLOGICAL E LIMITED\nAmount In INR</td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr>\n<tr><td>Description</td><td>SAC Code</td><td>Taxable V

In [20]:
from chonkie import TableChunker

table = """
'\n    <html>\n    <head>\n    <meta charset="UTF-8">\n    <style>\n    body { font-family: Arial, sans-serif; white-space: pre-wrap; }\n    table { border-collapse: collapse; width: 100%; margin: 8px 0; }\n    td, th { border: 1px solid #ccc; padding: 4px; text-align: left; }\n    </style>\n    </head>\n    <body>\n    <p>Tax</p>\n<p>Invoice</p>\n<p>(Original</p>\n<p>For</p>\n<p>Recipient)</p>\n<p>AIR</p>\n<p>INDIA</p>\n<p>EXPRESS</p>\n<p>LIMITED</p>\n<p>Address</p>\n<p>:</p>\n<p>2nd</p>\n<p>Floor,</p>\n<p>Airlines</p>\n<p>House,</p>\n<p>NSCBI</p>\n<p>Airport,</p>\n<p>39</p>\n<p>C</p>\n<p>R</p>\n<p>Avenue</p>\n<p>Kolkata,</p>\n<p>Kolkata,</p>\n<p>West</p>\n<p>Bengal,</p>\n<p>700052</p>\n<p>GSTN</p>\n<p>:</p>\n<p>19AABCA0522B1ZH</p>\n<p>Invoice</p>\n<p>Number</p>\n<p>:</p>\n<p>ICCU250700002431</p>\n<p>Invoice</p>\n<p>Date</p>\n<p>:</p>\n<p>02-07-2025</p>\n<p>PNR</p>\n<p>No</p>\n<p>:</p>\n<p>F6L6WW</p>\n<p>Passenger</p>\n<p>Name</p>\n<p>:</p>\n<p>Atanu</p>\n<p>Ghosh</p>\n<p>Passenger</p>\n<p>Address</p>\n<p>:</p>\n<p>GSTIN</p>\n<p>of</p>\n<p>Customer</p>\n<p>:</p>\n<p>36AAACB7873P1Z0</p>\n<p>GSTIN</p>\n<p>Customer</p>\n<p>Name</p>\n<p>:</p>\n<p>BIOLOGICAL</p>\n<p>E</p>\n<p>LIMITED</p>\n<p>Flight</p>\n<p>No</p>\n<p>:</p>\n<p>2826</p>\n<p>Flight</p>\n<p>Date</p>\n<p>:</p>\n<p>28-07-2025</p>\n<p>Flight</p>\n<p>From</p>\n<p>:</p>\n<p>CCU</p>\n<p>Flight</p>\n<p>To</p>\n<p>:</p>\n<p>HYD</p>\n<p>Place</p>\n<p>of</p>\n<p>Supply</p>\n<p>:</p>\n<p>TELANGANA</p>\n<p>[36]</p>\n<p>Amount</p>\n<p>In</p>\n<p>INR</p>\n<p>Description</p>\n<p>SAC</p>\n<p>Code</p>\n<p>Taxable</p>\n<p>Value /Exempt</p>\n<p>Non</p>\n<p>Taxable Value</p>\n<p>Total</p>\n<p>IGST Total</p>\n<p>Invoice Value</p>\n<p>Rate</p>\n<p>(%)</p>\n<p>Amount</p>\n<p>(Rs.)</p>\n<p>Air</p>\n<p>Ticket</p>\n<p>charges</p>\n<p>996425</p>\n<p>4,625.71</p>\n<p>-</p>\n<p>4,625.71</p>\n<p>5</p>\n<p>%</p>\n<p>231.29</p>\n<p>4,857.00</p>\n<p>Airport</p>\n<p>Taxes-Pass</p>\n<p>Through</p>\n<p>-</p>\n<p>-</p>\n<p>996.00</p>\n<p>996.00</p>\n<p>0</p>\n<p>%</p>\n<p>-</p>\n<p>996.00</p>\n<p>Grand</p>\n<p>Total</p>\n<p>4,625.71</p>\n<p>996.00</p>\n<p>5,621.71</p>\n<p>231.29</p>\n<p>5,853.00</p>\n<p>Declaration</p>\n<p>:</p>\n<p>"I/We</p>\n<p>hereby</p>\n<p>declare</p>\n<p>that</p>\n<p>though</p>\n<p>our</p>\n<p>aggregate</p>\n<p>turnover</p>\n<p>in</p>\n<p>any</p>\n<p>preceding</p>\n<p>financial</p>\n<p>year</p>\n<p>from</p>\n<p>2017-18</p>\n<p>onwards</p>\n<p>is</p>\n<p>more</p>\n<p>than</p>\n<p>the</p>\n<p>aggregate</p>\n<p>turnover</p>\n<p>notified</p>\n<p>under sub-rule</p>\n<p>(4)</p>\n<p>of</p>\n<p>rule</p>\n<p>48,</p>\n<p>we</p>\n<p>are</p>\n<p>not</p>\n<p>required</p>\n<p>to</p>\n<p>prepare</p>\n<p>an</p>\n<p>invoice</p>\n<p>in</p>\n<p>terms</p>\n<p>of</p>\n<p>the</p>\n<p>provisions</p>\n<p>of</p>\n<p>the</p>\n<p>said</p>\n<p>sub-rule."</p>\n<p>Whether</p>\n<p>Tax</p>\n<p>is</p>\n<p>payable</p>\n<p>on</p>\n<p>Reverse</p>\n<p>Charge</p>\n<p>:</p>\n<p>NO Note</p>\n<p>: (a)</p>\n<p>Air</p>\n<p>Ticket</p>\n<p>charges</p>\n<p>:</p>\n<p>It</p>\n<p>includes</p>\n<p>all</p>\n<p>the</p>\n<p>charges</p>\n<p>related</p>\n<p>to</p>\n<p>air</p>\n<p>travel. (b)</p>\n<p>Ancillary</p>\n<p>charges</p>\n<p>:</p>\n<p>It</p>\n<p>includes</p>\n<p>ancillary</p>\n<p>services</p>\n<p>related</p>\n<p>to</p>\n<p>air</p>\n<p>travel. (c)</p>\n<p>Airport</p>\n<p>Taxes-Pass</p>\n<p>Through</p>\n<p>:</p>\n<p>These</p>\n<p>are</p>\n<p>the</p>\n<p>charges</p>\n<p>collected</p>\n<p>on</p>\n<p>behalf</p>\n<p>of</p>\n<p>airport</p>\n<p>authority(PSF,ADF,UDF</p>\n<p>etc).</p>\n<p>AIR</p>\n<p>INDIA</p>\n<p>EXPRESS</p>\n<p>is</p>\n<p>a</p>\n<p>Pure</p>\n<p>Agent</p>\n<p>for</p>\n<p>these</p>\n<p>charges (d)</p>\n<p>Insurance-Pass</p>\n<p>through</p>\n<p>:</p>\n<p>Insurance</p>\n<p>charges</p>\n<p>are</p>\n<p>collected</p>\n<p>on</p>\n<p>behalf</p>\n<p>of</p>\n<p>the</p>\n<p>Insurance</p>\n<p>provider.</p>\n<p>AIR</p>\n<p>INDIA</p>\n<p>EXPRESS</p>\n<p>is</p>\n<p>a</p>\n<p>Pure</p>\n<p>Agent</p>\n<p>for</p>\n<p>these</p>\n<p>charges</p>\n<p>Registered</p>\n<p>Office</p>\n<p>:</p>\n<p>AIR</p>\n<p>INDIA</p>\n<p>EXPRESS</p>\n<p>LIMITED,</p>\n<p>Block</p>\n<p>4,</p>\n<p>Vatika</p>\n<p>One</p>\n<p>on</p>\n<p>One</p>\n<p>Sector</p>\n<p>–</p>\n<p>16,</p>\n<p>NH</p>\n<p>–</p>\n<p>48,</p>\n<p>Industrial</p>\n<p>Estate,</p>\n<p>Gurgoan</p>\n<p>–</p>\n<p>122007</p>\n<p>Haryana</p>\n<p>Corporate</p>\n<p>Office</p>\n<p>:</p>\n<p>AIR</p>\n<p>INDIA</p>\n<p>EXPRESS</p>\n<p>LIMITED,</p>\n<p>Block</p>\n<p>4,</p>\n<p>Vatika</p>\n<p>One</p>\n<p>on</p>\n<p>One</p>\n<p>Sector</p>\n<p>–</p>\n<p>16,</p>\n<p>NH</p>\n<p>–</p>\n<p>48,</p>\n<p>Industrial</p>\n<p>Estate,</p>\n<p>Gurgoan</p>\n<p>–</p>\n<p>122007</p>\n<p>Haryana</p>\n<p>CIN</p>\n<p>:</p>\n<p>U62100HR1971PLC113015</p>\n<p>,</p>\n<p>PAN</p>\n<p>:</p>\n<p>AABCA0522B</p>\n<p>Authorised</p>\n<p>Signatory.</p>\n<p>AIR</p>\n<p>INDIA</p>\n<p>EXPRESS</p>\n<p>LIMITED</p>\n</body></html>'
"""

chunker = TableChunker(tokenizer="row", chunk_size=3)
chunks = chunker.chunk(table)
for chunk in chunks:
	print(chunk.text)

# Each chunk is a valid markdown table segment, always including the header. For the example above and `chunk_size=3`, you might get:
# >>>
# | Name   | Age | City     |
# |--------|-----|----------|
# | Alice  | 30  | New York |
# | Bob    | 25  | London   |
# | Carol  | 28  | Paris    |

# | Name   | Age | City     |
# |--------|-----|----------|
# | Dave   | 35  | Berlin   |


'
    <html>
    <head>
    <meta charset="UTF-8">
    <style>
    body { font-family: Arial, sans-serif; white-space: pre-wrap; }
    table { border-collapse: collapse; width: 100%; margin: 8px 0; }
    td, th { border: 1px solid #ccc; padding: 4px; text-align: left; }
    </style>
    </head>
    <body>
    <p>Tax</p>
<p>Invoice</p>
<p>(Original</p>
<p>For</p>
<p>Recipient)</p>
<p>AIR</p>
<p>INDIA</p>
<p>EXPRESS</p>
<p>LIMITED</p>
<p>Address</p>
<p>:</p>
<p>2nd</p>
<p>Floor,</p>
<p>Airlines</p>
<p>House,</p>
<p>NSCBI</p>
<p>Airport,</p>
<p>39</p>
<p>C</p>
<p>R</p>
<p>Avenue</p>
<p>Kolkata,</p>
<p>Kolkata,</p>
<p>West</p>
<p>Bengal,</p>
<p>700052</p>
<p>GSTN</p>
<p>:</p>
<p>19AABCA0522B1ZH</p>
<p>Invoice</p>
<p>Number</p>
<p>:</p>
<p>ICCU250700002431</p>
<p>Invoice</p>
<p>Date</p>
<p>:</p>
<p>02-07-2025</p>
<p>PNR</p>
<p>No</p>
<p>:</p>
<p>F6L6WW</p>
<p>Passenger</p>
<p>Name</p>
<p>:</p>
<p>Atanu</p>
<p>Ghosh</p>
<p>Passenger</p>
<p>Address</p>
<p>:</p>
<p>GSTIN</p>
<p>of</p>
<p>Cust

In [2]:
!pip install chonkie


   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   -------------------- ------------------- 1/2 [chonkie]
   ---------------------------------------- 2/2 [chonkie]

